In [2]:
from transformers import PreTrainedModel, PretrainedConfig, AutoModelForCausalLM, AutoModel, AutoProcessor, AutoTokenizer
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from transformers.modeling_outputs import CausalLMOutputWithPast
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import os
import json
from PIL import Image
from typing import List, Dict, Any

In [13]:
class VLMConfig(PretrainedConfig):
    model_type = "vlm_model"
    def __init__(self,
                 llm_model_path = "Qwen2.5-0.5B-Instruct",
                 vision_model_path = "siglip-base-patch16-224",
                 freeze_vision_model = True,
                 image_pad_num = 49,
                 **kwargs):
        self.llm_model_path = llm_model_path # 定义语言模型路径
        self.vision_model_path = vision_model_path # 定义视觉模型路径
        self.freeze_vision_model = freeze_vision_model # 冻结视觉模型
        self.image_pad_num = image_pad_num # 每张图片分配 49 个 token
        super().__init__(**kwargs)

In [18]:
class VLM(PreTrainedModel):
    config_class = VLMConfig
    def __init__(self, config):
        super().__init__(config)
        self.config = config

        self.llm_model = AutoModelForCausalLM.from_pretrained(self.config.llm_model_path) # 加载语言模型
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.llm_model_path) # 加载 Qwen 的文字处理器
        self.vision_model = AutoModel.from_pretrained(self.config.vision_model_path) # 加载视觉模型
        self.processor = AutoProcessor.from_pretrained(self.config.vision_model_path) # 加载 SigLIP 的图像理器

        """
        视觉向文本对其
        首先将视觉的 hidden_size (模型本身参数里面 hidden size 的 4 倍，因为我们规定每个图片有 49 个 token，所以原本 4 个 token 合并成一个，hidden size 变为原本的 4 倍) 向文本的 hidden_size 对齐
        然后再通过一层线形层增强模型的表达能力
        """
        self.linear1 = nn.Linear(self.vision_model.config.vision_config.hidden_size * 4, self.llm_model.config.hidden_size)
        self.linear2 = nn.Linear(self.llm_model.config.hidden_size, self.llm_model.config.hidden_size)
        if self.config.freeze_vision_model: # 冻结两个模型的权重，在预训练过程中只训练两个用于对齐的全连接层的权重
            for param in self.vision_model.parameters():
                param.requires_grad = False
        for param in self.llm_model.parameters():
            param.requires_grad = False

    def forward(self, input_ids, labels, pixel_values, attention_mask=None):
        # 获取语言模型的 embedding 层并将文本的 input ids 输入得到对应的向量表示 (words embedding)
        # 文本数据 embedding 相对简单，只需调用 embedding 层查表即可
        text_embeds = self.llm_model.get_input_embeddings()(input_ids)

        # 获取视觉模型的 embedding
        # 图像的 embedding 相对复杂，要通过视觉模型的 forward 计算获得，所以要 vision_model() 调用向前传播，取 last_hidden_state
        image_embeds = self.vision_model.vision_model(pixel_values).last_hidden_state
        b, s, d = image_embeds.shape # (batch size, sequence length, token dimension)
        image_embeds = image_embeds.view(b, -1, d*4) # (b, 196, d) -> (b, 49, d*4) 压缩图片 token，减少序列长度
        image_features = self.linear2(F.silu(self.linear1(image_embeds))) # 视觉向文本对齐 hidden_dim

        text_embeds = text_embeds.to(image_features.dtype)

        input_embeds = self.merge_input_ids_with_image_features(image_features, text_embeds, input_ids) # 拼接文本和图像
        outputs = self.llm_model(inputs_embeds=input_embeds, attention_mask=attention_mask) # 得到输出
        logits = outputs[0] # (batch size, sequence length, vocab size)
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id) # 设置计算损失时，忽略填充用 token
            """
            logits (batch_size, sequence_length, vocab_size) -> (batch_size * sequence_length, vocab_size)
            labels (batch_size, sequence_length) -> (batch_size * sequence_length) 和 logits 第一维对应
            """
            loss = loss_fn(
                logits.view(-1, logits.size(-1)), labels.view(-1).to(logits.device)
            )
        return CausalLMOutputWithPast(loss=loss, logits=logits)

    def merge_input_ids_with_image_features(self, image_features, inputs_embeds, input_ids): # 将文本 token 和图像 token 合并
        num_images, num_image_patches, embed_dim = image_features.shape # (num_images, num_image_batch, embed_dim) 相当于 (b, t, c)
        # 获取 <|image_pad|> 的 token 值，然后通过 input_ids = token_id 生成一个布尔张量，通过 where 返回两个位置坐标张量
        batch_indices, image_indices = torch.where(input_ids == self.tokenizer('<|image_pad|>')['input_ids'][0])

        # 将 image_feature 展平，替换到 inputs_embeds 中对应的 <|image_pad|> 处
        inputs_embeds[batch_indices, image_indices] = image_features.view(-1, embed_dim)
        return inputs_embeds

In [19]:
class MyDataset(Dataset):
    def __init__(self, image_path, data_path, tokenizer, processor, config):
        self.image_path = image_path # 图片数据路径
        self.data_path = data_path # 文本数据路径
        self.tokenizer = tokenizer
        self.processor = processor
        self.config = config
        with open(self.data_path, 'r', encoding='utf-8') as f:
            self.datas = json.load(f) # 读取文本数据（JSON 格式）

    def __len__(self):
        return len(self.datas)

    def __getitem__(self, index):
        sample = self.datas[index]
        try:
            image_name = sample['image'] # 获取文本对应图片名
            conversations = sample['conversations'] # 获取对话
            """
            <|im_start|>system
            You are a helpful assistant.
            <|im_end|>
            <|im_start|>user
            你的问题内容
            <|im_end|>
            <|im_start|>assistant
            """
            q_text = self.tokenizer.apply_chat_template( # 将文本数据转化为 Qwen 的模版形式输入
                [
                    {"role":"system", "content":'You are a helpful assistant.'},
                    {"role":"user", "content":conversations[0]['value']}
                ],
                tokenize=False,
                add_generation_prompt=True # 在对话格式化后添加 <|im_start|>assistant，指示模型合适开始生成自己的内容
            ).replace('<image>', '<|image_pad|>'*self.config.image_pad_num) # 替换特殊符，并重复 image_pad_num 次，表示要占这么多个 token
            a_text = conversations[1]['value'] + self.tokenizer.eos_token # 回答文本
            q_input_ids = self.tokenizer(q_text)['input_ids'] # 获取问题 token ids
            a_input_ids = self.tokenizer(a_text)['input_ids'] # 获取回答 token ids
            input_ids = q_input_ids + a_input_ids # 拼接
            # 创建一个与问题长度相同的列表，全部填充特殊占位符，再将回答的token ids拼接到填充标记后面
            labels = [self.tokenizer.pad_token_id] * len(q_input_ids) + a_input_ids
            # 错位训练，输入移除开头一个 token，标签移除末尾一个 token，使模型在每个时间步都会基于之前所有token来预测下一个token
            input_ids = input_ids[:-1]
            labels = labels[1:]

            image = Image.open(os.path.join(self.image_path, image_name)).convert('RGB')
            pixel_values = self.processor(text=None, images=image)['pixel_values']
        except:
            default_image = Image.new('RGB', (224, 224), color='white')
            pixel_values = self.processor(text=None, images=default_image)['pixel_values']
            q_text = self.tokenizer.apply_chat_template(
                [{"role":"system", "content":'You are a helpful assistant.'}, {"role":"user", "content":"图片内容是什么\n<image>"}],
                tokenize=False,
                add_generation_prompt=True
            ).replace('<image>', '<|image_pad|>'*self.config.image_pad_num)
            a_text = '图片内容为空' + self.tokenizer.eos_token
            q_input_ids = self.tokenizer(q_text)['input_ids']
            a_input_ids = self.tokenizer(a_text)['input_ids']
            input_ids = q_input_ids + a_input_ids
            labels = [self.tokenizer.pad_token_id] * len(q_input_ids) + a_input_ids
            input_ids = input_ids[:-1]
            labels = labels[1:]
        return {
            'input_ids': input_ids,
            'labels': labels,
            'pixel_values': pixel_values
        }

In [20]:
class MyDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        max_len = max(len(feature['input_ids']) for feature in features) # 计算 input_ids 最大的序列长度，作为标准对齐所有序列
        input_ids = []
        labels = []
        pixel_values = []
        for feature in features:
            input_ids.append(feature['input_ids'] + [self.tokenizer.pad_token_id] * (max_len - len(feature['input_ids'])))
            labels.append(feature['labels'] + [self.tokenizer.pad_token_id] * (max_len - len(feature['labels'])))
            pixel_values.append(feature['pixel_values'])

        return {'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
                'pixel_values': torch.cat(pixel_values, dim=0)} # 将所有 pixel values 沿第一维度即 batch 堆叠起来

In [21]:
config = VLMConfig()
model = VLM(config)
print(model)
print(f'model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
images_path = "LLaVA-CC3M-Pretrain-595K/images"
data_path = "Chinese-LLaVA-Vision-Instructions/LLaVA-CC3M-Pretrain-595K/chat-translated.json"
tokenizer = AutoTokenizer.from_pretrained(config.llm_model_path)
processor = AutoProcessor.from_pretrained(config.vision_model_path)
output_dir = "save/pretrained"
args = TrainingArguments(
    output_dir=output_dir,
    do_train=True,
    per_device_train_batch_size=8,
    learning_rate=1e-4,
    num_train_epochs=5,
    save_steps=500,
    save_total_limit=2,
    fp16=False,
    gradient_accumulation_steps=8,
    logging_steps=100,
    report_to='tensorboard',
    dataloader_pin_memory=True,
    dataloader_num_workers=0
)
trainer = Trainer(
    model = model,
    args=args,
    train_dataset=MyDataset(images_path, data_path, tokenizer, processor, config),
    data_collator=MyDataCollator(tokenizer)
)

trainer.train(resume_from_checkpoint=False)
trainer.save_model('save/pretrained')
trainer.save_state()

VLM(
  (llm_model): Qwen2ForCausalLM(
    (model): Qwen2Model(
      (embed_tokens): Embedding(151936, 896)
      (layers): ModuleList(
        (0-23): 24 x Qwen2DecoderLayer(
          (self_attn): Qwen2SdpaAttention(
            (q_proj): Linear(in_features=896, out_features=896, bias=True)
            (k_proj): Linear(in_features=896, out_features=128, bias=True)
            (v_proj): Linear(in_features=896, out_features=128, bias=True)
            (o_proj): Linear(in_features=896, out_features=896, bias=False)
            (rotary_emb): Qwen2RotaryEmbedding()
          )
          (mlp): Qwen2MLP(
            (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
            (up_proj): Linear(in_features=896, out_features=4864, bias=False)
            (down_proj): Linear(in_features=4864, out_features=896, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
          (post_attention_layernorm): Qwen2RMSNorm((

Step,Training Loss


KeyboardInterrupt: 